In [1]:
from sympy import symbols, Eq, solve
import re
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

token = os.environ.get("HUGGING_TOKEN")

In [3]:
from transformers import pipeline

In [4]:
generator = pipeline("text-generation", model="mistralai/Ministral-8B-Instruct-2410", token=token, temperature=0.001)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
Device set to use cuda:0


In [12]:

prompt = "a plus b equals c. c equals 2 times b. and a equals c plus 1"
instruction = [
    {
        "role": "system", "content":
            "You are a natural language equation parser.\n"
            "1. If the input describes an inequality (>, <, >=, <=, !=, or their verbal forms), respond ONLY with: INEQUAL_WARNING.\n"
            "2. If the input does not describe a valid math equation, respond ONLY with: NOTMATH_WARNING.\n"
            "3. Otherwise, output ONLY a comma-separated list of equations. Each equation must:\n"
            "   - Be in single quotes: 'example'\n"
            "   - Have spaces around operators (+, -, *, /, **)\n"
            "   - Use '.' for decimal points\n"
            "   - Contain only = (no inequalities)\n"
            "Do not solve the equations. Do not explain anything. Output nothing except what the rules above require."
    },
    {"role": "user", "content": f"{prompt}\n"},
]


In [13]:
result = generator(instruction)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [14]:
def result_parser (result) :
    answer = result[0]["generated_text"] [2] ['content']

    answers = answer.split(', ')
    answers_cleared = []

    for elem in answers :
        elem_clr = elem.replace("'", "")

        answers_cleared.append(elem_clr)

    return answers_cleared


In [15]:
def explicit_multiplication (equation) :
    import re

    pattern = r'([0-9])([A-Za-z])|([A-Za-z])([0-9])'

    def insert_multiply(match):
        if match.group(1) and match.group(2):
            # digit+letter
            return f"{match.group(1)} * {match.group(2)}"
        else:
            # letter+digit
            return f"{match.group(3)} * {match.group(4)}"

    eq_expl = re.sub(pattern, insert_multiply, equation)
    return eq_expl

In [16]:
parsed_answer = result_parser(result)

expl_mult_equations = []
for eq in parsed_answer :
    expl_mult_equations.append(explicit_multiplication(eq))

print(expl_mult_equations)

['c = a + b', 'c = 2 * b', 'a = c + 1']


In [17]:
import re

def is_safe_equation(s: str) -> bool:
    if s.count('=') != 1: return False # only one equation sign
    if re.search(r"[\"'`_<>!^&|:%,$\\\[\]{}]", s): return False # forbidden chars
    if not re.fullmatch(r"[A-Za-z0-9+\-*/=().\s]+", s): return False # allowed chars
    if re.search(r"[A-Za-z]\s*\.\s*[A-Za-z0-9]", s): return False # a.b not allowed
    if re.search(r"[A-Za-z][A-Za-z0-9]*\s*\(", s): return False # not allowed fun(
    L, R = (p.strip() for p in s.split('='))
    if not L or not R: return False # something on both sides of equation

    bal = 0
    for ch in s: # both brackets present
        bal += (ch == '(') - (ch == ')')
        if bal < 0: return False
    return bal == 0

In [18]:
for eq in expl_mult_equations :
    if not is_safe_equation(eq) :
        raise Exception(f"The equation {eq} is not safe.")